# Treinamento do Modelo

- Treinar modelo
- Exportar NIR

# Exportar Dataset de Teste

- Salvar dataset como .npz
  - chave 'data' para os dados
  - chave 'labels' para os rótulos

## Exemplo com Torch

In [2]:
# import torch
# from torchvision import datasets, transforms
# from torch.utils.data import DataLoader
# import numpy as np

# # Transformação: converte PIL Images para tensores e normaliza
# transform = transforms.Compose([
#     transforms.ToTensor(),             # converte para torch.Tensor
#     transforms.Normalize((0.1307,), (0.3081,))  # normalização MNIST
# ])

# # Baixa MNIST de treino
# mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# # Cria DataLoader
# dataloader_torch = DataLoader(mnist_train, batch_size=1, shuffle=True)

In [3]:
# def salvar_dataset_npz(loader, arquivo):
#     dados = []
#     labels = []

#     for batch in loader:
#         # formato (x, y)
#         x, y = batch

#         # PyTorch
#         if torch.is_tensor(x):
#             x = x.cpu().numpy()
#             y = y.cpu().numpy()

#         dados.append(x)
#         labels.append(y)

#     dados = np.concatenate(dados, axis=0)
#     labels = np.concatenate(labels, axis=0)

#     np.savez_compressed(
#         arquivo,
#         data=dados,
#         labels=labels
#     )

#     print(f"Arquivo '{arquivo}.npz' salvo.")

In [4]:
# salvar_dataset_npz(dataloader_torch, "data_test.npz")

## Exemplo com Tensorflow

- TODO: FINALIZAR

In [5]:
# import tensorflow as tf

# # Carrega MNIST direto do Keras datasets
# (mnist_x_train, mnist_y_train), _ = tf.keras.datasets.mnist.load_data()

# # Normaliza os pixels
# mnist_x_train = mnist_x_train.astype('float32') / 255.0
# mnist_x_train = mnist_x_train[..., tf.newaxis]  # adiciona canal (28,28,1)

# # Converte labels para tf.int32
# mnist_y_train = mnist_y_train.astype('int32')

# # Cria tf.data.Dataset
# dataset_tf = tf.data.Dataset.from_tensor_slices((mnist_x_train, mnist_y_train))
# dataset_tf = dataset_tf.shuffle(buffer_size=10000).batch(32)

In [6]:
# import numpy as np

# def processar_batch(batch_x, batch_y):
#     """
#     Função genérica para processar um batch de dados.
#     Aqui você pode colocar treino, validação, ou qualquer operação.
#     """
#     print("Batch X shape:", batch_x.shape)
#     print("Batch Y shape:", batch_y.shape)
#     # Exemplo de operação simples
#     return batch_x.mean(), batch_y.mean()

# def iterar_dataloader(dataloader):
#     """
#     Itera sobre qualquer dataloader PyTorch ou TensorFlow,
#     convertendo para numpy arrays.
#     """
#     for batch_x, batch_y in dataloader:
#         # PyTorch e TensorFlow 2.x retornam tensors com .numpy()
#         if hasattr(batch_x, "numpy"):
#             batch_x = batch_x.numpy()
#             batch_y = batch_y.numpy()
#         # Agora batch_x e batch_y são np.array
#         yield batch_x, batch_y
        
# def main(dataloader_torch, dataloader_tf):
#     print("Iterando DataLoader do PyTorch")
#     for batch_x, batch_y in iterar_dataloader(dataloader_torch):
#         processar_batch(batch_x, batch_y)
    
#     print("\nIterando DataLoader do TensorFlow")
#     for batch_x, batch_y in iterar_dataloader(dataloader_tf):
#         processar_batch(batch_x, batch_y)

In [7]:
# main(dataloader_torch, dataset_tf)

# NeuroHls

In [1]:
from neuro_hls import *

In [2]:
neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")

## Definindo a Implementação do Modelo

In [3]:
# nir_file = "nir_examples/lif_norse.nir"
nir_file = "nir_examples/cnn_sinabs.nir"
# nir_file = "nir_examples/braille_noDelay_bias_zero.nir"

In [4]:
model = neuro_hls.read_nir_file(nir_file)

In [12]:
print(model)

-------------------------------------------------------
Input ([ 2 34 34]) - layer name: 'input'
-------------------------------------------------------
	Is recurrent: NO
	Dependencies:

-------------------------------------------------------
Conv2d (input: [ 2 34 34], output: [16 16 16]) - layer name: '0'
-------------------------------------------------------
	Weight shape: (16, 2, 5, 5)
	Stride: [2 2], Padding: [1 1], Dilation: [1 1]
	Groups: 1, Bias shape: (16,)
	Is recurrent: NO
	Dependencies:
	   - input (ready)

-------------------------------------------------------
IF (input: [16 16 16], output: [16 16 16]) - layer name: '1'
-------------------------------------------------------
	Parameter shape: (16, 16, 16)
	r range: [1.0000, 1.0000]
	v_threshold range: [1.0000, 1.0000]
	v_reset range: [0.0000, 0.0000]
	Is recurrent: NO
	Dependencies:
	   - 0 (ready)

-------------------------------------------------------
Conv2d (input: [16 16 16], output: [16 16 16]) - layer name: '2'
---

In [ ]:
neuro_hls.implement_model(model, use_float=False, use_event_driven=True)

## Criando o Testbench

In [6]:
# neuro_hls.define_test_dataset("nir_examples/rnn_teste.pt", data_is_binary=True, step_count=256, different_sample_per_step=True)
neuro_hls.define_test_dataset("nir_examples/n-mnist-150-steps.npz", data_is_binary=True, step_count=150, different_sample_per_step=True)

In [7]:
neuro_hls.create_testbench(total_samples=10, batch_size=10, reset_potentials=True)

Total samples used: 10 of 1000
Batch size: 10
Total batches: 1
Testbench was created.


# C-sim

In [ ]:
import importlib
import neuro_hls as neuro_hls_module
importlib.reload(neuro_hls_module)
from neuro_hls import NeuroHls

neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")
neuro_hls.run_csim()


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-08:29:27
  **** Start of session at: Mon Apr 20 14:49:02 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257

Sourcing Tcl script 'c:/Users/Mateus/NeuroHLS_dev/z_test/wrapper_csim.tcl'

INFO: [HLS 200-1510] Running: open_project vitis_proj 


Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html

INFO: [HLS 200-10] Opening solution 'C:/Users/Mateus/NeuroHLS_dev/z_test/vitis_proj'.

INFO: [HLS 200-1510] Running: open_solution sol 

INFO: [HLS 200-10] Opening solution 'C:/Users/Mateus/NeuroHLS_dev/z_test/vitis_proj/sol'.

INFO: [SYN 201-201] Setting up clock 'default_clk' with a period of 6.66667ns.

INFO: [HLS 200-1611] Setting target device to 'xcu250-figd2104-2L-e'

INFO: [HLS 200-1505] U

In [19]:
from neuro_hls import *
neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")
neuro_hls.run_synth(frequency_MHz=150, part="xcu250-figd2104-2L-e")


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-08:29:27
  **** Start of session at: Mon Apr 20 13:45:46 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257

Sourcing Tcl script 'c:/Users/Mateus/NeuroHLS_dev/z_test/wrapper_synth.tcl'

INFO: [HLS 200-1510] Running: open_project vitis_proj 


Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html

INFO: [HLS 200-10] Opening solution 'C:/Users/Mateus/NeuroHLS_dev/z_test/vitis_proj'.

INFO: [HLS 200-1510] Running: set_top snn_to_hls 

INFO: [HLS 200-1510] Running: open_solution sol 

INFO: [HLS 200-10] Opening solution 'C:/Users/Mateus/NeuroHLS_dev/z_test/vitis_proj/sol'.

INFO: [SYN 201-201] Setting up clock 'default_clk' with a period of 6.66667ns.

INFO: [HLS 200-1611] Setting target devi

In [3]:
neuro_hls.get_synth_resource_usage()

{'BRAM_18K': 10, 'DSP': 2680, 'FF': 268748, 'LUT': 663074, 'URAM': 0}

In [4]:
neuro_hls.get_synth_performance_estimates()

{'avg_total_cycles': 6041, 'avg_latency': '40.273 us'}

In [1]:
import importlib
import neuro_hls as neuro_hls_module
importlib.reload(neuro_hls_module)
from neuro_hls import NeuroHls

neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")
neuro_hls.run_cosim()


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-08:29:27
  **** Start of session at: Mon Apr 20 15:40:45 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257

Sourcing Tcl script 'c:/Users/Mateus/NeuroHLS_dev/z_test/wrapper_cosim.tcl'

INFO: [HLS 200-1510] Running: open_project vitis_proj 


Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html

INFO: [HLS 200-10] Opening solution 'C:/Users/Mateus/NeuroHLS_dev/z_test/vitis_proj'.

INFO: [HLS 200-1510] Running: open_solution sol 

INFO: [HLS 200-10] Opening solution 'C:/Users/Mateus/NeuroHLS_dev/z_test/vitis_proj/sol'.

INFO: [SYN 201-201] Setting up clock 'default_clk' with a period of 6.66667ns.

INFO: [HLS 200-1611] Setting target device to 'xcu250-figd2104-2L-e'

INFO: [HLS 200-1505] 

# Power report

In [ ]:
neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")

# 1. Configura a Cosim (sem executar) e Exporta para o Vivado
neuro_hls.configure_cosim_and_export()

In [2]:
neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")

# 2. Faz a Síntese no Vivado
neuro_hls.run_vivado_synthesis()


*** [Step 2] Running Vivado Synthesis ***

****** Vivado v2025.2 (64-bit)
  **** SW Build 6299465 on Fri Nov 14 12:34:56 MST 2025
  **** IP Build 6300035 on Fri Nov 14 10:48:45 MST 2025
  **** SharedData Build 6298862 on Thu Nov 13 04:50:51 MST 2025
  **** Start of session at: Mon Apr 27 19:35:12 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

source /home/rcarlos/Documentos/SNN/Renan/NeuroHLS_dev/z_test/vitis_proj/sol/impl/verilog/run_ooc_synth.tcl
# create_project -in_memory -part xcu250-figd2104-2L-e
# set vivado_ip_dir /home/rcarlos/Documentos/SNN/Renan/NeuroHLS_dev/z_test/vitis_proj/sol/impl/ip/hdl/verilog
# set hdl_files [glob -nocomplain -directory $vivado_ip_dir *.v]
# if {![llength $hdl_files]} { error "No exported HLS HDL files were found" }
# set_property include_dirs $vivado_ip_dir [current_fileset]
# read_verilog -sv $hdl_files
# read_xdc /home/rcarlos/Documentos/SNN/Renan/Neur

In [3]:
import importlib
import neuro_hls as neuro_hls_module
importlib.reload(neuro_hls_module)
from neuro_hls import NeuroHls

neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")

# 3. Gera o SAIF na simulação pós-síntese
neuro_hls.generate_post_synth_saif()


*** [Step 3] Generating Post-Synth SAIF Simulation ***

*** Unable to build post-synthesis SAIF artifacts: [Errno 8] Exec format error: '/home/rcarlos/Documentos/SNN/Renan/NeuroHLS_dev/z_test/vitis_proj/sol/sim/verilog/run_xsim_saif.bat'


In [4]:
import importlib
import neuro_hls as neuro_hls_module
importlib.reload(neuro_hls_module)
from neuro_hls import NeuroHls

neuro_hls = NeuroHls(folder_path="z_test", settings64_path=r"C:\AMDDesignTools\2025.2\Vitis\settings64.bat")

# 4. Gera o report de Power
neuro_hls.generate_power_report()


*** [Step 4] Generating Power Report from SAIF (post_synth=True) ***

*** Unable to run power report from SAIF: Missing or empty SAIF file: /home/rcarlos/Documentos/SNN/Renan/NeuroHLS_dev/z_test/vitis_proj/sol/impl/verilog/post_synth_activity.saif
